# Class 2 - Zero-Shot, Few-Shot & Role-Based Prompting

**Week 4: Prompt Engineering (Basic to Intermediate)**

### Learning objectives
By the end of this notebook you will be able to:
- Write a zero-shot prompt and recognize when it's not enough.
- Build a few-shot prompt that teaches a model a specific output pattern through examples.
- Assign a persona via a system prompt and explain what it does (and doesn't) change.
- Add basic chain-of-thought reasoning to a prompt and compare it against a direct answer.
- Choose the right technique (or combination of techniques) for a given task.


## Setup

Run this cell first. It reads your API key from the environment - never hardcode a key in a notebook.

Set one of these in your shell before launching Jupyter:

```bash
export OPENAI_API_KEY="sk-..."
# or
export ANTHROPIC_API_KEY="sk-ant-..."
```

You'll also need the matching SDK installed:

```bash
pip install openai anthropic
```


In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not OPENAI_API_KEY and not ANTHROPIC_API_KEY:
    print(
        "No API key found in the environment.\n"
        "Set one of these before running the demo cells below:\n"
        "  export OPENAI_API_KEY='sk-...'\n"
        "  export ANTHROPIC_API_KEY='sk-ant-...'\n"
        "The code below will run as soon as a key is set - nothing is hardcoded here."
    )
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"Found an API key for {provider}. You're ready to run the demo cells below.")

# --- Minimal provider-agnostic chat helper -------------------------------
# Uses OpenAI if OPENAI_API_KEY is set, otherwise falls back to Anthropic.

def ask(system_prompt, user_prompt, model=None, temperature=0.7, max_tokens=400):
    """Send a system + user prompt to whichever provider has a key set.
    Returns the assistant's text reply as a plain string."""
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        model = model or "gpt-4o-mini"
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        return resp.choices[0].message.content
    elif ANTHROPIC_API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        model = model or "claude-3-5-haiku-20241022"
        resp = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}],
        )
        return resp.content[0].text
    else:
        raise RuntimeError(
            "No API key set. Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your environment first."
        )


## Concept 1 - Zero-Shot Prompting

Zero-shot means no examples - just an instruction. It's the fastest thing to write, and often enough
for common, well-defined tasks like sentiment classification.

In [ ]:
review = "The battery life is amazing but the app crashes constantly."
zero_shot_prompt = f"Classify the sentiment of this review as positive, negative, or neutral:\n\n{review}"

print(ask("You are a sentiment classification assistant.", zero_shot_prompt))


## Concept 2 - Few-Shot Prompting

Few-shot prompting shows 2-5 input/output examples before the real task. The model completes the
pattern instead of following a described rule - useful when the exact format matters.

In [ ]:
few_shot_prompt = """Convert each input into a short news headline in title case.

Input: "Sales grew twenty percent this quarter"
Output: "Q3 Sales Surge 20%"

Input: "The company hired 40 new engineers"
Output: "Company Adds 40 Engineers"

Input: "Customer complaints dropped by half"
Output:"""

print(ask("You are a headline writer.", few_shot_prompt))


## Concept 3 - Zero-Shot vs. Few-Shot, Side by Side

Same reviews, two techniques. The few-shot version also demonstrates a specific output *format*
(`LABEL: ... | REASON: ...`) that would be hard to describe in words alone.

In [ ]:
reviews = [
    "The battery life is amazing but the app crashes constantly.",
    "Shipping was slow, but I love the product itself.",
    "Perfect in every way, would buy again.",
]

zero_shot_instr = "Classify sentiment as positive, negative, or neutral."

few_shot_instr = """Classify sentiment using this exact format: LABEL: <positive|negative|neutral> | REASON: <one short phrase>

Review: "Great quality but arrived a week late."
LABEL: neutral | REASON: mixed shipping and quality experience

Review: "Terrible, broke on day one."
LABEL: negative | REASON: product failure
"""

for r in reviews:
    print("Review:", r)
    print("Zero-shot:", ask("You are a sentiment classifier.", f"{zero_shot_instr}\n\n{r}"))
    print("Few-shot: ", ask("You are a sentiment classifier.", f'{few_shot_instr}\nReview: "{r}"\n'))
    print()


## Concept 4 - Role-Based / Persona Prompting

A persona set in the system prompt changes *tone and framing*, not facts. Same code, same question,
two very different-sounding reviews.

In [ ]:
code_snippet = """def login(username, password):
    query = f"SELECT * FROM users WHERE username=\'{username}\' AND password=\'{password}\'"
    return db.execute(query)
"""

personas = {
    "Security engineer": "You are a senior security engineer reviewing code for junior developers. Be direct, flag risks first, explain briefly.",
    "Friendly mentor": "You are a friendly coding mentor helping a beginner feel encouraged while still learning.",
}

for label, sys_prompt in personas.items():
    print(f"--- {label} ---")
    print(ask(sys_prompt, f"Review this login function for issues:\n\n{code_snippet}"))
    print()


## Concept 5 - Basic Chain-of-Thought

Asking the model to "think step by step" before answering often improves accuracy on multi-step
problems, at the cost of a longer response.

In [ ]:
math_problem = (
    "A train leaves at 2pm going 60mph. Another leaves the same station at 3pm "
    "going 90mph in the same direction. At what time does the second train catch the first?"
)

direct = ask("You are a math tutor.", math_problem)
cot = ask(
    "You are a math tutor.",
    math_problem + "\n\nThink through this step by step, then give your final answer "
    "on the last line, prefixed with 'ANSWER:'.",
)

print("--- Direct ---")
print(direct)
print("\n--- Chain-of-thought ---")
print(cot)


## Challenges

Work through these on your own. No solutions are provided - write your own prompts and code below each prompt.


### Challenge 1 - Zero-Shot First

Classify the three new reviews below zero-shot. For each, note (in a comment or printed line) anything
that seems wrong or ambiguous about the result.

**Acceptance criteria:** all three reviews classified, with at least one observation written per review.

In [ ]:
new_reviews = [
    "Not bad, not great. It does the job.",
    "Wow. Just... wow. Never buying this again.",
    "Exceeded every expectation I had.",
]

# TODO: classify each review zero-shot with ask(), and note anything ambiguous about the results.


### Challenge 2 - Build a Few-Shot Set

Pick a task of your own (e.g., converting bug reports into one-line summaries, or turning notes into
action items). Write 3 consistent input/output examples, then test the pattern on a new input.

**Acceptance criteria:** 3 examples with a consistent format, plus one new input the model completes correctly.

In [ ]:
# TODO: write your own few-shot prompt (3 examples + 1 new input) and call ask() with it.
my_few_shot_prompt = ""


### Challenge 3 - Design a Persona

Write a system prompt persona for a specific audience of your choosing (for example: "explain to a
5-year-old" or "explain to a CFO who doesn't code"). Ask it the same question and compare tone against
the "Neutral" persona from Class 1.

**Acceptance criteria:** one custom persona, one shared question, output printed and compared.

In [ ]:
# TODO: write your own persona system prompt and test it against a question of your choice.
my_persona = ""
my_question = ""


### Challenge 4 - Add Chain-of-Thought

Here's a logic puzzle. Compare a direct answer to a chain-of-thought answer, the same way Concept 5 did.

**Acceptance criteria:** both a direct and a chain-of-thought version run, with a short note on which was more accurate.

In [ ]:
logic_puzzle = (
    "Three friends split a $60 bill unevenly. Alex paid twice what Sam paid. "
    "Jordan paid $10 more than Sam. How much did each person pay?"
)

# TODO: run logic_puzzle both directly and with a chain-of-thought instruction, then compare.


### Challenge 5 - Combine All Three

Write ONE prompt that combines a persona (system prompt), few-shot examples, and a chain-of-thought
instruction, for a task of your choice.

**Acceptance criteria:** a single system prompt (persona) + a single user prompt containing both examples
and a "think step by step" instruction, run together with `ask()`.

In [ ]:
# TODO: combine a persona, few-shot examples, and a chain-of-thought instruction into one prompt.
combined_system_prompt = ""
combined_user_prompt = ""
